# TP1 – Continual Learning en Seq-CIFAR-10
**I309 – Visión Artificial Avanzada · Universidad de San Andrés**

Este notebook contiene todos los experimentos del TP, organizados en las 4 etapas del enunciado:

1. Preparación del dataset (Seq-CIFAR-10, replay buffer)
2. Pre-entrenamiento con Supervised Contrastive Learning (SupCon)
3. Métodos de aprendizaje continuo: Naive, EWC, LwF, Co²L
4. Comparación de resultados (Class-IL y Task-IL)

## 0. Setup

In [ ]:
import os

# Colab: clonar el repo y montar Drive para persistir checkpoints
if os.path.exists("/content"):
    if not os.path.exists("/content/Continual-Learning-on-Seq-CIFAR-10"):
        !git clone https://github.com/trinidadpujol/Continual-Learning-on-Seq-CIFAR-10.git
    %cd /content/Continual-Learning-on-Seq-CIFAR-10

    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # Checkpoints se guardan en Drive para sobrevivir reinicios de sesión
    CKPT_DRIVE = "/content/drive/MyDrive/TP1_checkpoints"
    os.makedirs(CKPT_DRIVE, exist_ok=True)

    # Sincronizar checkpoints existentes de Drive al runtime local
    !cp -n {CKPT_DRIVE}/*.pt checkpoints/ 2>/dev/null || true
    !cp -n {CKPT_DRIVE}/*.json checkpoints/ 2>/dev/null || true
    print(f"Checkpoints sincronizados desde Drive → checkpoints/")

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # avoid OpenMP conflict on Windows

import sys, os

# __file__ doesn't exist in Jupyter use the working directory instead.
# Launch Jupyter from the repo root (the folder containing tp1.ipynb) and
# this will resolve correctly.
REPO_ROOT = os.path.abspath('')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'REPO_ROOT: {REPO_ROOT}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')

PyTorch  : 2.2.2+cpu
REPO_ROOT: c:\Users\Hp\OneDrive\Documentos\UdeSA\5_ano\Vision_artificial_avanzada\TP1\Continual-Learning-on-Seq-CIFAR-10
Device   : cpu


In [2]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

from data.dataset import SeqCIFAR10, TASK_CLASSES, CLASS_NAMES
from data.buffer import ReplayBuffer
from models.backbone import Backbone
from losses.supcon import SupConLoss
from losses.distillation import AsymmetricDistillationLoss
from methods.naive import NaiveFineTuning
from methods.ewc import EWC
from methods.lwf import LwF
from methods.co2l import Co2L
from methods.trainer import ContinualTrainer
from methods.pretrain import pretrain_supcon, train_linear_probe, load_pretrained_backbone
from utils.metrics import evaluate_class_il, evaluate_task_il, compute_forgetting, MetricsTracker
from utils.visualization import plot_accuracy_curve, plot_forgetting_curve, plot_comparison, plot_forgetting_heatmap, plot_embeddings, plot_loss_curve, plot_pretrain_loss, plot_embedding_stages, plot_probe_accuracy

print('All imports OK')

All imports OK


## Etapa 1 – Preparación del Dataset

Dividimos CIFAR-10 en 5 tareas secuenciales de 2 clases cada una. El modelo solo
puede ver los datos de la tarea actual durante el entrenamiento (más lo que haya
en el buffer de replay).

| Tarea | Clases               | Imágenes (train) |
|-------|----------------------|------------------|
| 0     | airplane, automobile | 10 000           |
| 1     | bird, cat            | 10 000           |
| 2     | deer, dog            | 10 000           |
| 3     | frog, horse          | 10 000           |
| 4     | ship, truck          | 10 000           |

In [3]:
DATA_ROOT   = './data'   # torchvision downloads CIFAR-10 here on first run
N_TASKS     = 5
BATCH_SIZE  = 128
BUFFER_SIZE = 200        # replay buffer capacity across all past tasks

seq_cifar = SeqCIFAR10(data_root=DATA_ROOT, n_tasks=N_TASKS, batch_size=BATCH_SIZE)
buffer    = ReplayBuffer(capacity=BUFFER_SIZE)

print(f'Task splits (class indices): {TASK_CLASSES}')
print()
for t in range(N_TASKS):
    names = seq_cifar.get_class_names(t)
    n_train = seq_cifar.task_size(t, train=True)
    n_test  = seq_cifar.task_size(t, train=False)
    print(f'  Task {t}: {names[0]:12s} + {names[1]:12s}  |  train={n_train:5d}  test={n_test:4d}')

Files already downloaded and verified
Files already downloaded and verified
Task splits (class indices): [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

  Task 0: airplane     + automobile    |  train=10000  test=2000
  Task 1: bird         + cat           |  train=10000  test=2000
  Task 2: deer         + dog           |  train=10000  test=2000
  Task 3: frog         + horse         |  train=10000  test=2000
  Task 4: ship         + truck         |  train=10000  test=2000


In [ ]:
# --- Verify class filtering is correct ---
print('Verifying class filtering...')
for t in range(N_TASKS):
    c0, c1 = seq_cifar.get_classes(t)
    loader = seq_cifar.get_task_loader(t, train=True)
    seen_classes = set()
    for _, labels in loader:
        seen_classes.update(labels.tolist())
    assert seen_classes == {c0, c1}, f'Task {t}: expected {{{c0},{c1}}}, got {seen_classes}'
    print(f'  Task {t} classes {seen_classes} = {seq_cifar.get_class_names(t)}')

print()
# Verify a SupCon loader returns (B, 2, C, H, W)
supcon_loader = seq_cifar.get_task_loader(0, train=True, supcon=True)
imgs, labels = next(iter(supcon_loader))
assert imgs.ndim == 5 and imgs.shape[1] == 2 and imgs.shape[2:] == (3, 32, 32), \
    f'Expected (B,2,3,32,32) but got {imgs.shape}'
print(f'SupCon loader batch shape: {imgs.shape}  â†’  (B={imgs.shape[0]}, 2 views, C=3, H=32, W=32)')
print()
print('All checks passed.')

Verifying class filtering...
  Task 0 OK â€” classes {0, 1} = ('airplane', 'automobile')
  Task 1 OK â€” classes {2, 3} = ('bird', 'cat')
  Task 2 OK â€” classes {4, 5} = ('deer', 'dog')
  Task 3 OK â€” classes {6, 7} = ('frog', 'horse')
  Task 4 OK â€” classes {8, 9} = ('ship', 'truck')

SupCon loader batch shape: torch.Size([128, 2, 3, 32, 32])  â†’  (B=128, 2 views, C=3, H=32, W=32)

All checks passed.


## Etapa 2 – Pre-entrenamiento con Supervised Contrastive Learning

Antes de arrancar con el escenario de aprendizaje continuo, pre-entrenamos el backbone
en la Tarea 0 (airplane/automobile) usando SupCon loss (Khosla et al., NeurIPS 2020).
La idea es que las representaciones aprendidas con objetivos contrastivos son más
transferibles que las aprendidas con entropía cruzada estándar.

Después del pre-entrenamiento, congelamos el encoder y entrenamos una cabeza de
clasificación lineal para verificar la calidad de las features.

In [ ]:
# Guardar checkpoints en Drive después del pre-entrenamiento
if os.path.exists("/content"):
    !cp checkpoints/*.pt {CKPT_DRIVE}/ 2>/dev/null || true
    print("Checkpoints copiados a Drive.")

In [ ]:
backbone_pretrained = Backbone(num_classes=10).to(DEVICE)

from methods.pretrain import pretrain_supcon, load_pretrained_backbone

SUPCON_EPOCHS    = 200
SUPCON_LR        = 0.5
SUPCON_TEMP      = 0.07
CHECKPOINT_DIR   = "checkpoints"
CHECKPOINT_EVERY = 50

SUPCON_CKPT = os.path.join(CHECKPOINT_DIR, "supcon_pretrained_task0.pt")

if os.path.exists(SUPCON_CKPT):
    print(f"Checkpoint encontrado, saltando pre-entrenamiento: {SUPCON_CKPT}")
    ckpt = load_pretrained_backbone(SUPCON_CKPT, backbone_pretrained, device=DEVICE)
    # Reconstruimos pretrain_history mínimo para que las celdas siguientes funcionen
    pretrain_history = {
        "loss_history":    ckpt.get("loss_history", []),
        "checkpoint_path": SUPCON_CKPT,
        "stage_paths": {
            "init":  os.path.join(CHECKPOINT_DIR, "supcon_stage_init_task0.pt"),
            "mid":   os.path.join(CHECKPOINT_DIR, "supcon_stage_mid_task0.pt"),
            "final": SUPCON_CKPT,
        },
    }
else:
    pretrain_history = pretrain_supcon(
        backbone         = backbone_pretrained,
        seq_cifar        = seq_cifar,
        task_id          = 0,
        n_epochs         = SUPCON_EPOCHS,
        lr               = SUPCON_LR,
        temperature      = SUPCON_TEMP,
        device           = DEVICE,
        checkpoint_dir   = CHECKPOINT_DIR,
        checkpoint_every = CHECKPOINT_EVERY,
        save_stages      = True,
    )
    print(f"Pre-entrenamiento terminado. Loss final: {pretrain_history['loss_history'][-1]:.4f}")

print(f"Checkpoint: {pretrain_history['checkpoint_path']}")

# Guardar en Drive automáticamente para sobrevivir reinicios de sesión
if os.path.exists("/content") and 'CKPT_DRIVE' in globals():
    import shutil, glob
    for f in glob.glob("checkpoints/supcon*.pt"):
        shutil.copy(f, CKPT_DRIVE)
    print(f"Checkpoints SupCon guardados en {CKPT_DRIVE}")

In [7]:
# ── SupCon loss curve ─────────────────────────────────────────────────────────
from utils.visualization import plot_pretrain_loss

plot_pretrain_loss(
    loss_history = pretrain_history["loss_history"],
    save_path    = "imgs/supcon_pretrain_loss.png",
    title        = "SupCon Pre-training Loss — Task 0 (airplane / automobile)",
)
print("Saved: imgs/supcon_pretrain_loss.png")


In [ ]:
# ── Embedding projections: init / mid / final ─────────────────────────────────
# Loads backbone snapshots saved during pre-training and runs t-SNE on
# Task 0 test features (airplane / automobile), producing a 1x3 side-by-side.
from utils.visualization import plot_embedding_stages

test_loader_viz = seq_cifar.get_task_loader(0, train=False)

plot_embedding_stages(
    stage_paths       = pretrain_history["stage_paths"],
    backbone_template = backbone_pretrained,
    loader            = test_loader_viz,
    device            = DEVICE,
    class_names       = CLASS_NAMES,
    save_path         = "imgs/supcon_embeddings_stages.png",
    n_samples         = 1000,
    method            = "tsne",
)
print("Saved: imgs/supcon_embeddings_stages.png")


In [ ]:
# ── Linear probe on Task 0 ────────────────────────────────────────────────────
LINEAR_EPOCHS = 100
LINEAR_LR     = 1.0

PROBE_CKPT = os.path.join(CHECKPOINT_DIR, "linear_probe_task0.pt")

if os.path.exists(PROBE_CKPT):
    print(f"Checkpoint encontrado, saltando linear probe: {PROBE_CKPT}")
    ckpt = torch.load(PROBE_CKPT, map_location=DEVICE)
    backbone_pretrained.load_state_dict(ckpt["backbone_state_dict"])
    backbone_pretrained.freeze_encoder()
    probe_history = {
        "loss_history":      ckpt.get("loss_history", []),
        "train_acc_history": ckpt.get("train_acc_history", []),
        "test_acc":          ckpt["test_acc"],
        "checkpoint_path":   PROBE_CKPT,
    }
else:
    probe_history = train_linear_probe(
        backbone       = backbone_pretrained,
        seq_cifar      = seq_cifar,
        task_id        = 0,
        n_epochs       = LINEAR_EPOCHS,
        lr             = LINEAR_LR,
        device         = DEVICE,
        checkpoint_dir = CHECKPOINT_DIR,
    )
    if os.path.exists("/content") and 'CKPT_DRIVE' in globals():
        import shutil
        shutil.copy(probe_history["checkpoint_path"], CKPT_DRIVE)
        print(f"Linear probe checkpoint guardado en {CKPT_DRIVE}")

print(f"Encoder frozen: {backbone_pretrained.is_encoder_frozen}")
print(f"Linear probe — Task 0 test accuracy: {probe_history['test_acc']:.3f}")
print(f"Checkpoint: {probe_history['checkpoint_path']}")

In [ ]:
# ── Linear probe results ──────────────────────────────────────────────────────
from utils.visualization import plot_probe_accuracy

plot_probe_accuracy(
    probe_history = probe_history,
    save_path     = "imgs/linear_probe_curves.png",
    title         = "Linear Probe — Task 0",
)
print(f"Test accuracy: {probe_history['test_acc']:.3f}")
print("Saved: imgs/linear_probe_curves.png")


In [ ]:
import copy, json, os

# Partimos del backbone pre-entrenado con SupCon + linear probe sobre Tarea 0.
# Descongelamos el encoder para que todos los métodos puedan hacer fine-tuning completo.
backbone_pretrained.unfreeze_encoder()

def load_tracker_from_json(path, n_tasks=5):
    """Reconstruye un MetricsTracker desde el JSON guardado por save_tracker()."""
    with open(path) as f:
        data = json.load(f)
    t = MetricsTracker(n_tasks=n_tasks)
    for c_row, t_row, c_avg, t_avg in zip(
        data["class_il"]["acc_matrix"],
        data["task_il"]["acc_matrix"],
        data["class_il"]["avg_acc_curve"],
        data["task_il"]["avg_acc_curve"],
    ):
        t._class_il.append(c_row)
        t._task_il.append(t_row)
        t._avg_class_il.append(c_avg)
        t._avg_task_il.append(t_avg)
    return t

def make_eval_fn(seq_cifar, device, tracker):
    """Devuelve eval_fn para ContinualTrainer.

    Después de cada tarea t, evalúa sobre las tareas 0..t y registra en el tracker.
    """
    def eval_fn(task_id, method):
        test_loaders = [seq_cifar.get_task_loader(t, train=False)
                        for t in range(task_id + 1)]
        class_il = evaluate_class_il(method.backbone, test_loaders, device)
        task_il  = evaluate_task_il(method.backbone,  test_loaders, device)
        tracker.update(task_id, class_il, task_il)
        return {"class_il": class_il["avg_acc"], "task_il": task_il["avg_acc"]}
    return eval_fn

def save_tracker(tracker, name):
    """Guarda los resultados del tracker en checkpoints/<name>_results.json."""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    data = {
        "class_il": {
            "acc_matrix":      tracker._class_il,
            "avg_acc_curve":   tracker._avg_class_il,
            "forgetting":      tracker.forgetting("class_il"),
            "avg_forgetting":  tracker.avg_forgetting("class_il"),
        },
        "task_il": {
            "acc_matrix":      tracker._task_il,
            "avg_acc_curve":   tracker._avg_task_il,
            "forgetting":      tracker.forgetting("task_il"),
            "avg_forgetting":  tracker.avg_forgetting("task_il"),
        },
    }
    path = os.path.join(CHECKPOINT_DIR, f"{name}_results.json")
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Results saved to {path}")
    return path

def _backup_to_drive(filename):
    """Copia un archivo de checkpoints/ a CKPT_DRIVE si estamos en Colab."""
    if os.path.exists("/content") and 'CKPT_DRIVE' in globals():
        import shutil
        src = os.path.join(CHECKPOINT_DIR, filename)
        shutil.copy(src, CKPT_DRIVE)
        print(f"{filename} guardado en {CKPT_DRIVE}")

print("Setup de Etapa 3 listo — encoder descongelado:", not backbone_pretrained.is_encoder_frozen)

In [ ]:
# ── 3.1 Naive Fine-Tuning ────────────────────────────────────────────────────
# Baseline / cota inferior: fine-tuning secuencial con CE, sin ningún mecanismo
# anti-olvido. Partimos de los pesos pre-entrenados con SupCon.
NAIVE_JSON = os.path.join(CHECKPOINT_DIR, "naive_results.json")

if os.path.exists(NAIVE_JSON):
    print(f"Resultados encontrados, saltando entrenamiento Naive: {NAIVE_JSON}")
    naive_tracker = load_tracker_from_json(NAIVE_JSON, n_tasks=N_TASKS)
else:
    naive_tracker  = MetricsTracker(n_tasks=N_TASKS)
    naive_backbone = copy.deepcopy(backbone_pretrained)
    naive = NaiveFineTuning(naive_backbone, device=DEVICE, lr=0.01)
    naive_trainer = ContinualTrainer(
        naive, seq_cifar,
        n_epochs = 50,
        eval_fn  = make_eval_fn(seq_cifar, DEVICE, naive_tracker),
    )
    naive_results = naive_trainer.train_all_tasks()
    save_tracker(naive_tracker, "naive")
    _backup_to_drive("naive_results.json")

print(naive_tracker)
naive_tracker.summary_df()

In [ ]:
# ── 3.2 EWC ──────────────────────────────────────────────────────────────────
# Penalización cuadrática ponderada por la diagonal de la Fisher.
# La Fisher se estima con los datos de la tarea t al terminar cada tarea.
# λ=400 fue el valor que mejor funcionó en nuestros experimentos.
EWC_JSON = os.path.join(CHECKPOINT_DIR, "ewc_results.json")

if os.path.exists(EWC_JSON):
    print(f"Resultados encontrados, saltando entrenamiento EWC: {EWC_JSON}")
    ewc_tracker = load_tracker_from_json(EWC_JSON, n_tasks=N_TASKS)
else:
    ewc_tracker  = MetricsTracker(n_tasks=N_TASKS)
    ewc_backbone = copy.deepcopy(backbone_pretrained)
    ewc = EWC(ewc_backbone, device=DEVICE, ewc_lambda=400.0)
    ewc_trainer = ContinualTrainer(
        ewc, seq_cifar,
        n_epochs = 50,
        eval_fn  = make_eval_fn(seq_cifar, DEVICE, ewc_tracker),
    )
    ewc_results = ewc_trainer.train_all_tasks()
    save_tracker(ewc_tracker, "ewc")
    _backup_to_drive("ewc_results.json")

print(ewc_tracker)
ewc_tracker.summary_df()

In [ ]:
# ── 3.3 LwF ──────────────────────────────────────────────────────────────────
# Antes de cada tarea t > 0 se hace un snapshot frozen del modelo (teacher).
# Durante el entrenamiento: L = CE(tarea nueva) + α * KL(teacher || student) * T²
# El factor T² re-escala el gradiente de los soft targets (Hinton et al. 2015).
LWF_JSON = os.path.join(CHECKPOINT_DIR, "lwf_results.json")

if os.path.exists(LWF_JSON):
    print(f"Resultados encontrados, saltando entrenamiento LwF: {LWF_JSON}")
    lwf_tracker = load_tracker_from_json(LWF_JSON, n_tasks=N_TASKS)
else:
    lwf_tracker  = MetricsTracker(n_tasks=N_TASKS)
    lwf_backbone = copy.deepcopy(backbone_pretrained)
    lwf = LwF(lwf_backbone, device=DEVICE, alpha=1.0, temperature=2.0)
    lwf_trainer = ContinualTrainer(
        lwf, seq_cifar,
        n_epochs = 50,
        eval_fn  = make_eval_fn(seq_cifar, DEVICE, lwf_tracker),
    )
    lwf_results = lwf_trainer.train_all_tasks()
    save_tracker(lwf_tracker, "lwf")
    _backup_to_drive("lwf_results.json")

print(lwf_tracker)
lwf_tracker.summary_df()

In [ ]:
# ── 3.4 Co²L ─────────────────────────────────────────────────────────────────
# Implementación completa de Cha et al. (ICCV 2021).
# L = L_SupCon(batch conjunto) + λ * L_distill(A-Distill) + L_CE(tarea actual)
# El buffer se actualiza al terminar cada tarea (reservoir sampling).
CO2L_JSON = os.path.join(CHECKPOINT_DIR, "co2l_results.json")

if os.path.exists(CO2L_JSON):
    print(f"Resultados encontrados, saltando entrenamiento Co²L: {CO2L_JSON}")
    co2l_tracker = load_tracker_from_json(CO2L_JSON, n_tasks=N_TASKS)
else:
    co2l_tracker  = MetricsTracker(n_tasks=N_TASKS)
    co2l_backbone = copy.deepcopy(backbone_pretrained)
    co2l_buffer   = ReplayBuffer(capacity=BUFFER_SIZE)
    co2l = Co2L(
        backbone      = co2l_backbone,
        device        = DEVICE,
        seq_cifar     = seq_cifar,
        replay_buffer = co2l_buffer,
        temperature   = 0.07,
        lr            = 0.5,
        n_buf_samples = 64,
    )
    co2l_trainer = ContinualTrainer(
        co2l, seq_cifar,
        n_epochs = 100,
        eval_fn  = make_eval_fn(seq_cifar, DEVICE, co2l_tracker),
    )
    co2l_results = co2l_trainer.train_all_tasks()
    save_tracker(co2l_tracker, "co2l")
    _backup_to_drive("co2l_results.json")

print(co2l_tracker)
co2l_tracker.summary_df()

## Etapa 4 – Comparación de Resultados

Agregamos los resultados de los 4 métodos y generamos:
- Curvas de accuracy promedio vs. tareas (Class-IL y Task-IL)
- Matrices de accuracy por tarea (para ver qué tareas se olvidan más)
- Tabla resumen con métricas finales y forgetting promedio
- Heatmaps de accuracy por tarea para cada método

In [ ]:
import pandas as pd, json as _json, os, math, numpy as np

TASK_NAMES = ["T0\nplane/car", "T1\nbird/cat", "T2\ndeer/dog",
              "T3\nfrog/horse", "T4\nship/truck"]
FORG_NAMES = TASK_NAMES[:-1]   # la última tarea no tiene forgetting

def get_tracker(name, n_tasks=N_TASKS):
    """Devuelve el tracker en memoria si existe, sino lo carga desde el checkpoint JSON."""
    live = globals().get(f"{name.lower().replace('²','2').replace(' ','_')}_tracker")
    if live is not None and len(live._avg_class_il) > 0:
        return live
    path = os.path.join(CHECKPOINT_DIR, f"{name.lower().replace('²','2')}_results.json")
    if os.path.exists(path):
        print(f"  Cargando {name} desde {path}")
        return load_tracker_from_json(path, n_tasks)
    raise FileNotFoundError(
        f"No hay tracker en memoria ni checkpoint en {path}. "
        f"Ejecutar primero la celda de Etapa 3 para {name}."
    )

all_trackers = {}
for _name in ("Naive", "EWC", "LwF", "Co²L"):
    try:
        all_trackers[_name] = get_tracker(_name)
        print(f"  {_name}: {all_trackers[_name]}")
    except FileNotFoundError as e:
        print(f"  ADVERTENCIA: {e}")

print(f"\nMétodos cargados: {list(all_trackers.keys())}")
assert all_trackers, "No se cargó ningún tracker — ejecutar al menos un método de Etapa 3 primero."
os.makedirs("imgs", exist_ok=True)

# Curvas de accuracy individuales
plot_accuracy_curve(
    {m: t.avg_acc_curve("class_il") for m, t in all_trackers.items()},
    scenario="Class-IL",
    save_path="imgs/class_il_accuracy.png",
    task_names=TASK_NAMES,
)
plot_accuracy_curve(
    {m: t.avg_acc_curve("task_il") for m, t in all_trackers.items()},
    scenario="Task-IL",
    save_path="imgs/task_il_accuracy.png",
    task_names=TASK_NAMES,
    y_min=0.5,
)
plot_forgetting_curve(
    {m: t.forgetting("class_il") for m, t in all_trackers.items()},
    save_path="imgs/forgetting.png",
    task_names=FORG_NAMES,
)

# Figura de comparación unificada (3 paneles)
plot_comparison(
    all_trackers,
    save_path="imgs/comparison.png",
    task_names=TASK_NAMES,
    y_min_task_il=0.5,
)

# Heatmaps de accuracy por tarea
plot_forgetting_heatmap(
    all_trackers, scenario="class_il",
    save_path="imgs/heatmap_class_il.png",
    task_names=TASK_NAMES,
)
plot_forgetting_heatmap(
    all_trackers, scenario="task_il",
    save_path="imgs/heatmap_task_il.png",
    task_names=TASK_NAMES,
)

print("Imágenes guardadas:")
for p in ["imgs/class_il_accuracy.png", "imgs/task_il_accuracy.png",
          "imgs/forgetting.png", "imgs/comparison.png",
          "imgs/heatmap_class_il.png", "imgs/heatmap_task_il.png"]:
    print(f"  {p}")

In [ ]:
# ── Tabla resumen ─────────────────────────────────────────────────────────────
rows_summary = []
for method_name, tracker in all_trackers.items():
    c_curve = tracker.avg_acc_curve("class_il")
    t_curve = tracker.avg_acc_curve("task_il")
    rows_summary.append({
        "Method":             method_name,
        "Class-IL final":     round(c_curve[-1], 4),
        "Task-IL final":      round(t_curve[-1],  4),
        "Avg Forgetting (C)": round(tracker.avg_forgetting("class_il"), 4),
        "Avg Forgetting (T)": round(tracker.avg_forgetting("task_il"),  4),
    })

df_summary = pd.DataFrame(rows_summary).set_index("Method")

print("\n" + "="*60)
print("  Resultados finales — todos los métodos")
print("="*60)
print(df_summary.to_string())
print("="*60 + "\n")
df_summary

In [ ]:
# ── Accuracy por tarea después de entrenar las 5 tareas ──────────────────────
# Muestra qué tareas son más olvidadas por cada método
import math

for scenario in ("class_il", "task_il"):
    label = "Class-IL" if scenario == "class_il" else "Task-IL"
    rows_pt = []
    for method_name, tracker in all_trackers.items():
        mat = tracker.acc_matrix(scenario)     # (T, T) triángulo inferior
        final_row = mat[-1]                    # accuracy después de todas las tareas
        row = {"Method": method_name}
        for t, acc in enumerate(final_row):
            row[f"Task {t}"] = round(float(acc), 3) if not math.isnan(float(acc)) else float("nan")
        rows_pt.append(row)
    df_pt = pd.DataFrame(rows_pt).set_index("Method")
    print(f"\n{label} — Accuracy por tarea después de entrenar las 5 tareas:")
    print(df_pt.to_string())

print()
df_pt

In [ ]:
# ── Exportar resultados ───────────────────────────────────────────────────────
import numpy as np
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# CSV con la tabla resumen (para el informe)
csv_path = os.path.join(CHECKPOINT_DIR, "results_summary.csv")
df_summary.to_csv(csv_path)
print(f"Guardado: {csv_path}")

# JSON con curvas completas y matrices de accuracy de todos los métodos
full_export = {}
for method_name, tracker in all_trackers.items():
    mat_c = tracker.acc_matrix("class_il")
    mat_t = tracker.acc_matrix("task_il")
    full_export[method_name] = {
        "class_il": {
            "acc_matrix":     [[None if np.isnan(v) else round(float(v), 4)
                                 for v in row] for row in mat_c],
            "avg_acc_curve":  [round(v, 4) for v in tracker.avg_acc_curve("class_il")],
            "forgetting":     [round(v, 4) for v in tracker.forgetting("class_il")],
            "avg_forgetting": round(tracker.avg_forgetting("class_il"), 4),
        },
        "task_il": {
            "acc_matrix":     [[None if np.isnan(v) else round(float(v), 4)
                                 for v in row] for row in mat_t],
            "avg_acc_curve":  [round(v, 4) for v in tracker.avg_acc_curve("task_il")],
            "forgetting":     [round(v, 4) for v in tracker.forgetting("task_il")],
            "avg_forgetting": round(tracker.avg_forgetting("task_il"), 4),
        },
    }

json_path = os.path.join(CHECKPOINT_DIR, "results_full.json")
with open(json_path, "w") as f:
    _json.dump(full_export, f, indent=2)
print(f"Guardado: {json_path}")
print("\nExportación completa.")